# ITCS 6162: Data Mining - Programming Assignment

**In this assignment, you will explore data analysis, recommendation algorithms, and graph-based techniques using the MovieLens dataset. Your tasks will range from basic data exploration to advanced recommendation models, including:**
- Data manipulation with pandas
- User-item collaborative filtering
- Similarity-based recommendation models
- A Pixie-inspired Graph-based recommendation using adjacency lists with weighted random walks (without using NetworkX)


#### **Dataset Files:**
- **`u.data`**: User-movie ratings (`user_id  movie_id  rating  timestamp`)
- **`u.item`**: Movie metadata (`movie_id | title | release date | IMDB_website`)
- **`u.user`**: User demographics (`user_id | age | gender | occupation | zip_code`)

## **Part 1: Exploring and Cleaning Data**

### Inspecting the Dataset Format

The dataset is not in a traditional CSV format. To examine its structure, use the following shell command to display the first 10 lines of the file:

In [40]:
# u.data
#!head ../upload/u.data
file_path = r"F:\DATA MIN PROG ASS\u.data"

with open(file_path, 'r') as file:
    print("First 10 lines of u.data:")
    for _ in range(10):
        print(file.readline().strip())


First 10 lines of u.data:
196	242	3	881250949
186	302	3	891717742
22	377	1	878887116
244	51	2	880606923
166	346	1	886397596
298	474	4	884182806
115	265	2	881171488
253	465	5	891628467
305	451	3	886324817
6	86	3	883603013


In [6]:
# u.item
#!head ../upload/u.item
file_path = r"F:\DATA MIN PROG ASS\u.item"

with open(file_path, 'r', encoding='latin-1') as file:  # encoding to handle special characters
    print("First 10 lines of u.item:")
    for _ in range(10):
        print(file.readline().strip())


First 10 lines of u.item:
1|Toy Story (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)|0|0|0|1|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0
2|GoldenEye (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?GoldenEye%20(1995)|0|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
3|Four Rooms (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
4|Get Shorty (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995)|0|1|0|0|0|1|0|0|1|0|0|0|0|0|0|0|0|0|0
5|Copycat (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Copycat%20(1995)|0|0|0|0|0|0|1|0|1|0|0|0|0|0|0|0|1|0|0
6|Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)|01-Jan-1995||http://us.imdb.com/Title?Yao+a+yao+yao+dao+waipo+qiao+(1995)|0|0|0|0|0|0|0|0|1|0|0|0|0|0|0|0|0|0|0
7|Twelve Monkeys (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Twelve%20Monkeys%20(1995)|0|0|0|0|0|0|0|0|1|0|0|0|0|0|0|1|0|0|0
8|Babe (1995)|01-Jan-1995||http://us.imdb.com/M/title-exa

In [7]:
# u.user
#!head ../upload/u.user
file_path = r"F:\DATA MIN PROG ASS\u.user"

with open(file_path, 'r') as file:
    print("First 10 lines of u.user:")
    for _ in range(10):
        print(file.readline().strip())
    


First 10 lines of u.user:
1|24|M|technician|85711
2|53|F|other|94043
3|23|M|writer|32067
4|24|M|technician|43537
5|33|F|other|15213
6|42|M|executive|98101
7|57|M|administrator|91344
8|36|M|administrator|05201
9|29|M|student|01002
10|53|M|lawyer|90703


#### Loading the Dataset with Pandas

Use **pandas** to load the dataset into a DataFrame for analysis. Follow these steps:  

1. Import the necessary library: `pandas`.  
2. Use `pd.read_csv()` (or an appropriate function) to read the dataset file.  
3. Ensure the dataset is loaded with the correct delimiter (e.g., `','`, `'\t'`,`'|'` , or another separator if needed).  
4. Select and display the first few rows using `.head()`.

Ensure that:  

- The `ratings` dataset is read from `"u.data"` using tab (`'\t'`) as a separator and column names (`"user_id"`, `"movie_id"`, `"rating"` and `"timestamp"`).  
- The `movies` dataset is read from `"u.item"` using `'|'` as a separator, use columns (`0`, `1`, `2`), encoding (`"latin-1"`) and name the columns (`movie_id`, `title`, and `release_date`).  
- The `users` dataset is read from `"u.user"` using `'|'` as a separator, use columns (`0`, `1`, `2`, `3`) and name the columns (`user_id`, `age`, `gender`, and `occupation`).

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import random

In [10]:
# ratings
ratings = pd.read_csv('u.data', sep='\t', 
                      names=['user_id', 'movie_id', 'rating', 'timestamp'])
ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [11]:
# movies
movies = pd.read_csv('u.item', sep='|', encoding='latin-1', 
                     usecols=[0, 1, 2], names=['movie_id', 'title', 'release_date'])
movies.head()

,movie_id,title,release_date
0,1,Toy Story (1995),01-Jan-1995
1,2,GoldenEye (1995),01-Jan-1995
2,3,Four Rooms (1995),01-Jan-1995
3,4,Get Shorty (1995),01-Jan-1995
4,5,Copycat (1995),01-Jan-1995


In [12]:
# users
users = pd.read_csv('u.user', sep='|', 
                    usecols=[0, 1, 2, 3], names=['user_id', 'age', 'gender', 'occupation'])
users.head()

,user_id,age,gender,occupation
0,1,24,M,technician
1,2,53,F,other
2,3,23,M,writer
3,4,24,M,technician
4,5,33,F,other


**Note:** As a **Bonus** task save the `ratings`, `movies` and `users` dataframe created into a `.csv` file format.

In [13]:
# ratings
ratings.to_csv('ratings.csv', index=False)

In [14]:
# movies
movies.to_csv('movies.csv', index=False)

In [15]:
# users
users.to_csv('users.csv', index=False)

**Display the first 10 rows of each file.**

In [16]:
# ratings
ratings.head(10)

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
5,298,474,4,884182806
6,115,265,2,881171488
7,253,465,5,891628467
8,305,451,3,886324817
9,6,86,3,883603013


In [17]:
# movies
movies.head(10)

,movie_id,title,release_date
0,1,Toy Story (1995),01-Jan-1995
1,2,GoldenEye (1995),01-Jan-1995
2,3,Four Rooms (1995),01-Jan-1995
3,4,Get Shorty (1995),01-Jan-1995
4,5,Copycat (1995),01-Jan-1995
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,01-Jan-1995
6,7,Twelve Monkeys (1995),01-Jan-1995
7,8,Babe (1995),01-Jan-1995
8,9,Dead Man Walking (1995),01-Jan-1995
9,10,Richard III (1995),22-Jan-1996


In [18]:
# users
users.head(10)

,user_id,age,gender,occupation
0,1,24,M,technician
1,2,53,F,other
2,3,23,M,writer
3,4,24,M,technician
4,5,33,F,other
5,6,42,M,executive
6,7,57,M,administrator
7,8,36,M,administrator
8,9,29,M,student
9,10,53,M,lawyer


### Data Cleaning and Exploration with Pandas  

After loading the dataset, it’s important to clean and explore the data to ensure consistency and accuracy. Below are key **pandas** functions for cleaning and understanding the dataset.

#### 1. Handle Missing Values  
- `df.dropna()` – Removes rows with missing values.  
- `df.fillna(value)` – Fills missing values with a specified value.  

#### 2. Remove Duplicates  
- `df.drop_duplicates()` – Drops duplicate rows from the dataset.  

#### 3. Handle Incorrect Data Types  
- `df.astype(dtype)` – Converts columns to the appropriate data type.  

#### 4. Filter Outliers (if applicable)  
- `df[df['column_name'] > threshold]` – Filters rows based on a condition.  

#### 5. Rename Columns (if needed)  
- `df.rename(columns={'old_name': 'new_name'})` – Renames columns for clarity.  

#### 6. Reset Index  
- `df.reset_index(drop=True, inplace=True)` – Resets the index after cleaning.  

### Data Exploration Functions  

To better understand the dataset, use these **pandas** functions:  

- `df.shape` – Returns the number of rows and columns in the dataset.  
- `df.nunique()` – Displays the number of unique values in each column.  
- `df['column_name'].unique()` – Returns unique values in a specific column.  

**Example Usage in Pandas:**  
```python
import pandas as pd

# Load dataset
df = pd.read_csv("your_file.csv")

# Drop missing values
df_cleaned = df.dropna()

# Remove duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# Convert 'timestamp' column to datetime format
df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'])

# Display dataset shape
print("Dataset shape:", df_cleaned.shape)

# Display number of unique values in each column
print("Unique values per column:\n", df_cleaned.nunique())

# Display unique movie IDs
print("Unique movie IDs:", df_cleaned['movie_id'].unique()[:10])  # Show first 10 unique movie IDs


**Note:** The functions mentioned above are some of the widely used **pandas** functions for data cleaning and exploration. However, it is not necessary that all of these functions will be required in the exercises below. Use them as needed based on the dataset and the specific tasks.

**Convert Timestamps into Readable dates.**

In [19]:
# ratings
ratings['timestamp'] = ratings['timestamp'].apply(lambda x: datetime.fromtimestamp(x))
ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,1997-12-04 10:55:49
1,186,302,3,1998-04-04 15:22:22
2,22,377,1,1997-11-07 02:18:36
3,244,51,2,1997-11-27 00:02:03
4,166,346,1,1998-02-02 00:33:16


**Check for Missing Values**

In [20]:
# ratings
print("Missing values in ratings:")
print(ratings.isnull().sum())

Missing values in ratings:
user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64


In [21]:
# movies
print("Missing values in movies:")
print(movies.isnull().sum())

Missing values in movies:
movie_id        0
title           0
release_date    1
dtype: int64


In [22]:
# users
print("Missing values in users:")
print(users.isnull().sum())

Missing values in users:
user_id       0
age           0
gender        0
occupation    0
dtype: int64


**Print the total number of users, movies, and ratings.**

In [23]:
print(f"Total Users: {users['user_id'].nunique()}")
print(f"Total Movies: {movies['movie_id'].nunique()}")
print(f"Total Ratings: {len(ratings)}")

Total Users: 943
Total Movies: 1682
Total Ratings: 100000


## **Part 2: Collaborative Filtering-Based Recommendation**

### **Create a User-Item Matrix**

#### Instructions for Creating a User-Movie Rating Matrix

In this exercise, you will create a user-movie rating matrix using **pandas**. This matrix will represent the ratings that users have given to different movies.

1. **Dataset Overview**:  
   The dataset has already been loaded. It includes the following key columns:
   - `user_id`: The ID of the user.
   - `movie_id`: The ID of the movie.
   - `ratings`: The rating the user gave to the movie.

2. **Create the User-Movie Rating Matrix**:  
   Use the **`pivot()`** function in **pandas** to reshape the data. Your goal is to create a matrix where:
   - Each **row** represents a **user**.
   - Each **column** represents a **movie**.
   - Each **cell** contains the **rating** that the user has given to the movie.

   Specify the following parameters for the `pivot()` function:
   - **`index`**: The `user_id` column (this will define the rows).
   - **`columns`**: The `movie_id` column (this will define the columns).
   - **`values`**: The `rating` column (this will fill the matrix with ratings).

3. **Inspect the Matrix**:  
   After creating the matrix, examine the first few rows of the resulting matrix to ensure it has been constructed correctly.

4. **Handle Missing Values**:  
   It's likely that some users have not rated every movie, resulting in `NaN` values in the matrix. You will need to handle these missing values. Consider the following options:
   - **Fill with 0**: If you wish to represent missing ratings as zeros (indicating no rating).
   - **Fill with the average rating**: Alternatively, replace missing values with the average rating for each movie.

**Create the user-movie rating matrix using the `pivot()` function.**

In [24]:
user_movie_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating')
user_movie_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Display the matrix to verify the transformation.**

In [25]:
# Check the shape of the matrix
print(f"Matrix shape: {user_movie_matrix.shape}")

# Check the percentage of missing values
missing_percentage = user_movie_matrix.isnull().sum().sum() / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]) * 100
print(f"Percentage of missing values: {missing_percentage:.2f}%")

Matrix shape: (943, 1682)
Percentage of missing values: 93.70%


### **User-Based Collaborative Filtering Recommender System**

#### **Objective**
In this task, you will implement a **user-based collaborative filtering** movie recommendation system using the **Movie dataset**. The goal is to recommend movies to a user based on the preferences of similar users.

##### **Step 1: Import Required Libraries**
Before starting, ensure you have the necessary libraries installed. Use the following imports:

```python
import pandas as pd  # For handling data
import numpy as np   # For numerical computations
from sklearn.metrics.pairwise import cosine_similarity  # For computing user similarity
```

##### **Step 2: Compute User-User Similarity**
- We will use **cosine similarity** to measure how similar each pair of users is based on their movie ratings.
- Since `cosine_similarity` does not handle missing values (NaN), replace them with `0` before computation.

##### **Instructions:**
1. Fill missing values with `0` using `.fillna(0)`.
2. Compute similarity using `cosine_similarity()`.
3. Convert the result into a **Pandas DataFrame**, with users as both row and column labels.

##### **Hint:**  
You can achieve this using the following approach:

```python
user_similarity = cosine_similarity(user_movie_matrix.fillna(0))
user_sim_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)
```

##### **Step 3: Implement the Recommendation Function**
Now, implement the function `recommend_movies_for_user(user_id, num=5)` to recommend movies for a given user.

##### **Function Inputs:**
- `user_id`: The target user for whom we need recommendations.
- `num`: The number of movies to recommend (default is 5).

##### **Function Steps:**
1. Find **similar users**:
   - Retrieve the similarity scores for the given `user_id`.
   - Sort them in **descending** order (highest similarity first).
   - Exclude the user themselves.
   
2. Get the **movie ratings** from these similar users.

3. Compute the **average rating** for each movie based on these users' preferences.

4. Sort the movies in **descending order** based on the computed average ratings.

5. Retrieve the **top `num` recommended movies**.

6. Map **movie IDs** to their **titles** using the `movies` DataFrame.

7. Return the results as a **Pandas DataFrame** with rankings.

##### **Step 4: Return the Final Recommendation List**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

##### **Hint:** Your final DataFrame should be created like this:
```python
result_df = pd.DataFrame({
    'Ranking': range(1, num+1),
    'Movie Name': movie_names     
})
result_df.set_index('Ranking', inplace=True)
```

#### **Example: User-Based Collaborative Filtering**
```python
recommend_movies_for_user(10, num = 5)
```
**Output:**
```
| Ranking | Movie Name                     |
|---------|--------------------------------|
| 1       | In the Company of Men (1997)   |
| 2       | Misérables, Les (1995)         |
| 3       | Thin Blue Line, The (1988)     |
| 4       | Braindead (1992)               |
| 5       | Boys, Les (1997)               |


In [26]:
# Compute user-user similarity
user_similarity = cosine_similarity(user_movie_matrix.fillna(0))
user_sim_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# Display a sample of the similarity matrix
user_sim_df.iloc[:5, :5]

user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475
2,0.166931,1.000000,0.110591,0.178121,0.072979
3,0.047460,0.110591,1.000000,0.344151,0.021245
4,0.064358,0.178121,0.344151,1.000000,0.031804
5,0.378475,0.072979,0.021245,0.031804,1.000000


In [27]:
def recommend_movies_for_user(user_id, num=5):
    """
    Recommend movies for a user based on similar users' preferences.
    
    Parameters:
    user_id (int): The ID of the user for whom to make recommendations
    num (int): The number of movies to recommend
    
    Returns:
    DataFrame: A DataFrame with movie recommendations ranked by predicted rating
    """
    # Check if user exists
    if user_id not in user_sim_df.index:
        return f"User {user_id} not found in the dataset."
    
    # Get similarity scores for this user with all other users
    user_similarities = user_sim_df.loc[user_id].drop(user_id)
    
    # Sort similarities in descending order
    user_similarities = user_similarities.sort_values(ascending=False)
    
    # Get the movies that the user has already rated
    user_rated_movies = user_movie_matrix.loc[user_id].dropna().index
    
    # Initialize a dictionary to store weighted ratings
    weighted_ratings = {}
    similarity_sums = {}
    
    # For each similar user
    for similar_user, similarity in user_similarities.items():
        # Get the movies rated by this similar user
        similar_user_ratings = user_movie_matrix.loc[similar_user].dropna()
        
        # For each movie rated by the similar user
        for movie, rating in similar_user_ratings.items():
            # Skip if the target user has already rated this movie
            if movie in user_rated_movies:
                continue
                
            # Add weighted rating to the movie
            if movie not in weighted_ratings:
                weighted_ratings[movie] = 0
                similarity_sums[movie] = 0
                
            weighted_ratings[movie] += similarity * rating
            similarity_sums[movie] += similarity
    
    # Calculate the average weighted rating for each movie
    movie_predictions = {}
    for movie in weighted_ratings:
        if similarity_sums[movie] > 0:  # Avoid division by zero
            movie_predictions[movie] = weighted_ratings[movie] / similarity_sums[movie]
    
    # Sort movies by predicted rating
    sorted_predictions = sorted(movie_predictions.items(), key=lambda x: x[1], reverse=True)
    
    # Get the top N movies
    top_movies = sorted_predictions[:num]
    
    # Get movie names
    movie_names = []
    for movie_id, _ in top_movies:
        movie_title = movies.loc[movies['movie_id'] == movie_id, 'title'].values[0]
        movie_names.append(movie_title)
    
    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        'Ranking': range(1, len(movie_names) + 1),
        'Movie Name': movie_names
    })
    result_df.set_index('Ranking', inplace=True)
    
    return result_df

In [28]:
# Test the user-based recommendation function
recommend_movies_for_user(10, num=5)

,Movie Name
Ranking,
1,Santa with Muscles (1996)
2,Marlene Dietrich: Shadow and Light (1996)
3,"Great Day in Harlem, A (1994)"
4,They Made Me a Criminal (1939)
5,Someone Else's America (1995)


### **Item-Based Collaborative Filtering Recommender System**

#### **Objective**
In this task, you will implement an **item-based collaborative filtering** recommendation system using the **Movie dataset**. The goal is to recommend movies similar to a given movie based on user rating patterns.

#### **Step 1: Import Required Libraries**
Although we have done this part already in the previous task but just to emphasize the importance reiterrating this part.

Before starting, ensure you have the necessary libraries installed. Use the following imports:

```python
import pandas as pd  # For handling data
import numpy as np   # For numerical computations
from sklearn.metrics.pairwise import cosine_similarity  # For computing item similarity
```

#### **Step 2: Compute Item-Item Similarity**
- We will use **cosine similarity** to measure how similar each pair of movies is based on their user ratings.
- Since `cosine_similarity` does not handle missing values (NaN), replace them with `0` before computation.
- Unlike user-based filtering, we need to **transpose** (`.T`) the `user_movie_matrix` because we want similarity between movies (columns) instead of users (rows).

##### **Instructions:**
1. Transpose the user-movie matrix using `.T` to make movies the rows.
2. Fill missing values with `0` using `.fillna(0)`.
3. Compute similarity using `cosine_similarity()`.
4. Convert the result into a **Pandas DataFrame**, with movies as both row and column labels.

##### **Hint:**  
You can achieve this using the following approach:

```python
item_similarity = cosine_similarity(user_movie_matrix.T.fillna(0))
item_sim_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)
```

#### **Step 3: Implement the Recommendation Function**
Now, implement the function `recommend_movies(movie_name, num=5)` to recommend movies similar to a given movie.

##### **Function Inputs:**
- `movie_name`: The target movie for which we need recommendations.
- `num`: The number of similar movies to recommend (default is 5).

##### **Function Steps:**
1. Find the **movie_id** corresponding to the given `movie_name` in the `movies` DataFrame.
2. If the movie is not found, return an appropriate message.
3. Extract the **similarity scores** for this movie from `item_sim_df`.
4. Sort the movies in **descending order** based on similarity (excluding the movie itself).
5. Retrieve the **top `num` similar movies**.
6. Map **movie IDs** to their **titles** using the `movies` DataFrame.
7. Return the results as a **Pandas DataFrame** with rankings.

#### **Step 4: Return the Final Recommendation List**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

##### **Hint:** Your final DataFrame should be created like this:
```python
result_df = pd.DataFrame({
    'ranking': range(1, num+1),
    'movie_name': movie_names
})
result_df.set_index('ranking', inplace=True)
```

#### **Example: Item-Based Collaborative Filtering**
```python
recommend_movies("Jurassic Park (1993)", num=5)
```
**Output:**
```
| Ranking | Movie Name                               |
|---------|------------------------------------------|
| 1       | Top Gun (1986)                           |
| 2       | Empire Strikes Back, The (1980)          |
| 3       | Raiders of the Lost Ark (1981)           |
| 4       | Indiana Jones and the Last Crusade (1989)|
| 5       | Speed (1994)                             |


In [29]:
# Compute item-item similarity
item_similarity = cosine_similarity(user_movie_matrix.T.fillna(0))
item_sim_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

# Display a sample of the similarity matrix
item_sim_df.iloc[:5, :5]

movie_id,1,2,3,4,5
movie_id,,,,,
1,1.000000,0.402382,0.330245,0.454938,0.286714
2,0.402382,1.000000,0.273069,0.502571,0.318836
3,0.330245,0.273069,1.000000,0.324866,0.212957
4,0.454938,0.502571,0.324866,1.000000,0.334239
5,0.286714,0.318836,0.212957,0.334239,1.000000


In [30]:
def recommend_movies(movie_name, num=5):
    """
    Recommend movies similar to a given movie.
    
    Parameters:
    movie_name (str): The name of the movie for which to find similar movies
    num (int): The number of similar movies to recommend
    
    Returns:
    DataFrame: A DataFrame with similar movie recommendations ranked by similarity
    """
    # Find the movie_id for the given movie name
    movie_row = movies[movies['title'] == movie_name]
    
    if movie_row.empty:
        return f"Movie '{movie_name}' not found in the dataset."
    
    movie_id = movie_row.iloc[0]['movie_id']
    
    # Check if the movie exists in the similarity matrix
    if movie_id not in item_sim_df.index:
        return f"Movie '{movie_name}' (ID: {movie_id}) has no ratings in the dataset."
    
    # Get similarity scores for this movie with all other movies
    movie_similarities = item_sim_df.loc[movie_id].drop(movie_id)
    
    # Sort similarities in descending order
    movie_similarities = movie_similarities.sort_values(ascending=False)
    
    # Get the top N similar movies
    top_similar_movies = movie_similarities.head(num)
    
    # Get movie names
    movie_names = []
    for similar_movie_id in top_similar_movies.index:
        movie_title = movies.loc[movies['movie_id'] == similar_movie_id, 'title'].values[0]
        movie_names.append(movie_title)
    
    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        'Ranking': range(1, len(movie_names) + 1),
        'Movie Name': movie_names
    })
    result_df.set_index('Ranking', inplace=True)
    
    return result_df

In [31]:
# Test the item-based recommendation function
recommend_movies("Jurassic Park (1993)", num=5)

,Movie Name
Ranking,
1,Top Gun (1986)
2,Speed (1994)
3,Raiders of the Lost Ark (1981)
4,"Empire Strikes Back, The (1980)"
5,Indiana Jones and the Last Crusade (1989)


## **Part 3: Graph-Based Recommender (Pixie-Inspired Algorithm)**

### **Adjacency List**

#### **Objective**
In this task, you will preprocess the Movie dataset and construct a **graph representation** where:
- **Users** are connected to the movies they have rated.
- **Movies** are connected to users who have rated them.
  
This graph structure will help in exploring **user-movie relationships** for recommendations.

#### **Step 1: Merge Ratings with Movie Titles**
Since we have **movie IDs** in the ratings dataset but need human-readable movie titles, we will:
1. Merge the `ratings` DataFrame with the `movies` DataFrame using the `'movie_id'` column.
2. This allows each rating to be associated with a **movie title**.

#### **Hint:**
Use the following Pandas operation to merge:
```python
ratings = ratings.merge(movies, on='movie_id')
```


#### **Step 2: Aggregate Ratings**
Since multiple users may rate the same movie multiple times, we:
1. Group the dataset by `['user_id', 'movie_id', 'title']`.
2. Compute the **mean rating** for each movie by each user.
3. Reset the index to ensure we maintain a clean DataFrame structure.

#### **Hint:**  
Use `groupby()` and `mean()` as follows:
```python
ratings = ratings.groupby(['user_id', 'movie_id', 'title'])['rating'].mean().reset_index()
```

#### **Step 3: Normalize Ratings**
Since different users have different rating biases, we normalize ratings by:
1. **Computing each user's mean rating**.
2. **Subtracting the mean rating** from each individual rating.

#### **Instructions:**
- Use `groupby('user_id')` to group ratings by users.
- Apply `transform(lambda x: x - x.mean())` to adjust ratings.

#### **Hint:**  
Normalize ratings using:
```python
ratings['rating'] = ratings.groupby('user_id')['rating'].transform(lambda x: x - x.mean())
```
This ensures each user’s ratings are centered around zero, making similarity calculations fairer.

#### **Step 4: Construct the Graph Representation**
We represent the user-movie interactions as an **undirected graph** using an **adjacency list**:
- Each **user** is a node connected to movies they rated.
- Each **movie** is a node connected to users who rated it.

#### **Graph Construction Steps:**
1. Initialize an empty dictionary `graph = {}`.
2. Iterate through the **ratings dataset**.
3. For each `user_id` and `movie_id` pair:
   - Add the movie to the user’s set of connections.
   - Add the user to the movie’s set of connections.

#### **Hint:**  
The following code builds the graph:

```python
graph = {}
for _, row in ratings.iterrows():
    user, movie = row['user_id'], row['movie_id']
    if user not in graph:
        graph[user] = set()
    if movie not in graph:
        graph[movie] = set()
    graph[user].add(movie)
    graph[movie].add(user)
```

This results in a **bipartite graph**, where:
- **Users** are connected to multiple movies.
- **Movies** are connected to multiple users.

#### **Step 5: Understanding the Graph**
- **Nodes** in the graph represent **users and movies**.
- **Edges** exist between a user and a movie **if the user has rated the movie**.
- This structure allows us to find **users with similar movie tastes** and **movies frequently watched together**.

#### **Exploring the Graph**
- **Find a user’s rated movies:**  
  ```python
  user_id = 1
  print(graph[user_id])  # Movies rated by user 1
  ```

- **Find users who rated a movie:**  
  ```python
  movie_id = 50
  print(graph[movie_id])  # Users who rated movie 50
  ```

In [32]:
# Step 1: Merge Ratings with Movie Titles
ratings_with_titles = ratings.merge(movies, on='movie_id')
ratings_with_titles.head()

,user_id,movie_id,rating,timestamp,title,release_date
0,196,242,3,1997-12-04 10:55:49,Kolya (1996),24-Jan-1997
1,186,302,3,1998-04-04 15:22:22,L.A. Confidential (1997),01-Jan-1997
2,22,377,1,1997-11-07 02:18:36,Heavyweights (1994),01-Jan-1994
3,244,51,2,1997-11-27 00:02:03,Legends of the Fall (1994),01-Jan-1994
4,166,346,1,1998-02-02 00:33:16,Jackie Brown (1997),01-Jan-1997


In [33]:
# Step 2: Aggregate Ratings
aggregated_ratings = ratings_with_titles.groupby(['user_id', 'movie_id', 'title'])['rating'].mean().reset_index()
aggregated_ratings.head()

,user_id,movie_id,title,rating
0,1,1,Toy Story (1995),5.0
1,1,2,GoldenEye (1995),3.0
2,1,3,Four Rooms (1995),4.0
3,1,4,Get Shorty (1995),3.0
4,1,5,Copycat (1995),3.0


In [34]:
# Step 3: Normalize Ratings
aggregated_ratings['rating'] = aggregated_ratings.groupby('user_id')['rating'].transform(lambda x: x - x.mean())
aggregated_ratings.head()

,user_id,movie_id,title,rating
0,1,1,Toy Story (1995),1.389706
1,1,2,GoldenEye (1995),-0.610294
2,1,3,Four Rooms (1995),0.389706
3,1,4,Get Shorty (1995),-0.610294
4,1,5,Copycat (1995),-0.610294


In [35]:
# Step 4: Construct the Graph Representation
graph = {}

for _, row in aggregated_ratings.iterrows():
    user, movie = row['user_id'], row['movie_id']
    
    if user not in graph:
        graph[user] = set()
    if movie not in graph:
        graph[movie] = set()
        
    graph[user].add(movie)
    graph[movie].add(user)

# Display a sample of the graph
print(f"Total nodes in the graph: {len(graph)}")
print(f"Sample of user 1's connections: {list(graph[1])[:5]}")
print(f"Sample of movie 50's connections: {list(graph[50])[:5]}")

Total nodes in the graph: 1682
Sample of user 1's connections: [1, 2, 3, 4, 5]
Sample of movie 50's connections: [1, 2, 4, 5, 6]


### **Implement Weighted Random Walks**

#### **Random Walk-Based Movie Recommendation System (Weighted Pixie)**

#### **Objective**
In this task, you will implement a **random-walk-based recommendation algorithm** using the **Weighted Pixie** method. This technique uses a **user-movie bipartite graph** to recommend movies by simulating a random walk from a given user or movie.

#### **Step 1: Import Required Libraries**
Make sure you have the necessary libraries:

```python
import random  # For random walks
import pandas as pd  # For handling data
```

#### **Step 2: Implement the Random Walk Algorithm**
Your task is to **simulate a random walk** from a given starting point in the **bipartite user-movie graph**.

##### **Hints for Implementation**
- Start from **either a user or a movie**.
- At each step, **randomly move** to a connected node.
- Keep track of **how many times each movie is visited**.
- After completing the walk, **rank movies by visit count**.

#### **Step 3: Implement User-Based Recommendation**
**Hints:**
- Check if the `user_id` exists in the `graph`.
- Start a loop that runs for `walk_length` steps.
- Randomly pick a **connected node** (user or movie).
- Track how many times each **movie** is visited.
- Sort movies by visit frequency and return the **top N**.

#### **Step 4: Implement Movie-Based Recommendation**
**Hints:**
- Find the `movie_id` corresponding to the given `movie_name`.
- Ensure the movie exists in the `graph`.
- Start a random walk from that movie.
- Follow the same **tracking and ranking** process as the user-based version.

**Note:**  
**Your task:** Implement a function `weighted_pixie_recommend(user_id, walk_length=15, num=5)` or `weighted_pixie_recommend(movie_name, walk_length=15, num=5)`.  
**Implement either Step 3 or Step 4.**

#### **Step 5: Running Your Recommendation System**
Once your function is implemented, test it by calling:

##### **Example: User-Based Recommendation**
```python
weighted_pixie_recommend(1, walk_length=15, num=5)
```
| Ranking | Movie Name                     |
|---------|--------------------------------|
| 1       | My Own Private Idaho (1991)   |
| 2       | Aladdin (1992)                |
| 3       | 12 Angry Men (1957)           |
| 4       | Happy Gilmore (1996)          |
| 5       | Copycat (1995)                |


##### **Example: Movie-Based Recommendation**
```python
weighted_pixie_recommend("Jurassic Park (1993)", walk_length=10, num=5)
```
| Ranking | Movie Name                           |
|---------|-------------------------------------|
| 1       | Rear Window (1954)                 |
| 2       | Great Dictator, The (1940)         |
| 3       | Field of Dreams (1989)             |
| 4       | Casablanca (1942)                  |
| 5       | Nightmare Before Christmas, The (1993) |


#### **Step 6: Understanding the Results**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

Each movie is ranked based on **how frequently it was visited** during the walk.

#### **Experiment with Different Parameters**
- Try different **`walk_length`** values and observe how it changes recommendations.
- Adjust the number of recommended movies (`num`).

In [36]:
def weighted_pixie_recommend(start_point, walk_length=15, num=5):
    """
    Recommend movies using a weighted random walk on the user-movie graph.
    
    Parameters:
    start_point (int or str): Either a user_id or a movie name to start the walk from
    walk_length (int): The number of steps in each random walk
    num (int): The number of movies to recommend
    
    Returns:
    DataFrame: A DataFrame with movie recommendations ranked by visit frequency
    """
    # Determine if start_point is a user_id or a movie name
    if isinstance(start_point, str):
        # It's a movie name, find its ID
        movie_row = movies[movies['title'] == start_point]
        if movie_row.empty:
            return f"Movie '{start_point}' not found in the dataset."
        start_point = movie_row.iloc[0]['movie_id']
    
    # Check if the start point exists in the graph
    if start_point not in graph:
        return f"Start point {start_point} not found in the graph."
    
    # Dictionary to count movie visits
    movie_visits = {}
    
    # Perform the random walk
    current_node = start_point
    
    for _ in range(walk_length):
        # Get neighbors of the current node
        neighbors = list(graph[current_node])
        
        if not neighbors:
            break
        
        # Randomly select the next node
        next_node = random.choice(neighbors)
        current_node = next_node
        
        # If the current node is a movie, count it
        if current_node in movies['movie_id'].values:
            if current_node not in movie_visits:
                movie_visits[current_node] = 0
            movie_visits[current_node] += 1
    
    # If the start point is a movie, exclude it from recommendations
    if start_point in movies['movie_id'].values and start_point in movie_visits:
        del movie_visits[start_point]
    
    # Sort movies by visit count
    sorted_movies = sorted(movie_visits.items(), key=lambda x: x[1], reverse=True)
    
    # Get the top N movies
    top_movies = sorted_movies[:num]
    
    # If no movies were visited, return a message
    if not top_movies:
        return "No movie recommendations found. Try increasing the walk length."
    
    # Get movie names
    movie_names = []
    for movie_id, _ in top_movies:
        movie_title = movies.loc[movies['movie_id'] == movie_id, 'title'].values[0]
        movie_names.append(movie_title)
    
    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        'Ranking': range(1, len(movie_names) + 1),
        'Movie Name': movie_names
    })
    result_df.set_index('Ranking', inplace=True)
    
    return result_df

In [37]:
# Test the weighted pixie recommendation function with a user
weighted_pixie_recommend(1, walk_length=15, num=5)

,Movie Name
Ranking,
1,Delicatessen (1991)
2,Leaving Las Vegas (1995)
3,Maya Lin: A Strong Clear Vision (1994)
4,Batman & Robin (1997)
5,Grumpier Old Men (1995)


In [33]:
# Test the weighted pixie recommendation function with a movie
weighted_pixie_recommend("Jurassic Park (1993)", walk_length=10, num=5)

,Movie Name
Ranking,
1,"Garden of Finzi-Contini, The (Giardino dei Fin..."
2,Kama Sutra: A Tale of Love (1996)
3,Moll Flanders (1996)
4,Ridicule (1996)
5,Things to Do in Denver when You're Dead (1995)


---

## **Explanation of Pixie-Inspired Algorithms (3-5 Paragraphs)**

Pixie-inspired recommendation systems are graph-based algorithms that leverage the structure of user-item interactions to generate personalized recommendations. Unlike traditional collaborative filtering methods that rely on matrix operations, Pixie algorithms represent the recommendation problem as a bipartite graph where users and items are nodes, and interactions (such as ratings or views) form edges between them. This graph representation captures the complex relationships between users and items in an intuitive way, allowing for more nuanced recommendation strategies.

At the core of Pixie-inspired algorithms is the concept of random walks on graphs. A random walk is a stochastic process where, starting from a specific node (either a user or an item), the algorithm "walks" to neighboring nodes with certain probabilities. In the context of recommendation systems, these walks traverse between users and items, with the frequency of visits to item nodes serving as an indicator of relevance to the starting point. The key insight is that items frequently visited during these random walks are likely to be of interest to the user who initiated the walk, or similar to the item that served as the starting point.

What makes Pixie algorithms particularly powerful is their ability to incorporate various forms of weighting into the random walk process. Unlike simple random walks where transitions between nodes occur with uniform probability, weighted Pixie algorithms can adjust these probabilities based on factors such as rating scores, interaction recency, or item popularity. This weighting mechanism allows the algorithm to prioritize certain paths in the graph, leading to more relevant recommendations. Additionally, Pixie algorithms can be efficiently implemented even for very large graphs, making them suitable for real-world recommendation systems with millions of users and items.

In industry applications, Pixie-inspired algorithms have been successfully deployed by major technology companies. Pinterest, for example, developed the original Pixie algorithm to power their recommendation system, helping users discover relevant pins based on their interests and interactions. Similarly, companies like Netflix and Spotify use graph-based recommendation approaches to suggest movies and music to their users. These real-world implementations demonstrate the practical value of Pixie-inspired algorithms in enhancing user experience through personalized content discovery. The flexibility of these algorithms also allows them to be adapted to various domains beyond traditional e-commerce or content platforms, including social networks, job recommendations, and educational resource suggestions.

## **Movie Recommendation System Report**

### **1. Introduction**

Movie recommendation systems have become an essential component of online streaming platforms and entertainment services, helping users navigate through vast catalogs of content to find films that align with their preferences. These systems address the problem of information overload by filtering and prioritizing content based on user behavior and preferences. In this report, we explore three distinct approaches to movie recommendation: user-based collaborative filtering, item-based collaborative filtering, and random walk-based recommendations using a Pixie-inspired algorithm.

User-based collaborative filtering operates on the principle that users with similar taste patterns will likely enjoy similar movies. By identifying users with similar rating histories, the system can recommend movies that these similar users have enjoyed but the target user hasn't yet watched. Item-based collaborative filtering, on the other hand, focuses on the relationships between items rather than users. It identifies movies similar to those a user has already rated highly, based on how other users have rated these movies. Finally, the Pixie-inspired random walk approach represents users and movies as nodes in a bipartite graph, using stochastic processes to traverse this graph and identify relevant movie recommendations. Each of these approaches offers unique advantages and insights into user preferences, providing a comprehensive framework for movie recommendation.

### **2. Dataset Description**

The MovieLens 100K dataset, collected by the GroupLens Research Project at the University of Minnesota, serves as the foundation for our recommendation system implementation. This dataset contains 100,000 ratings (on a scale of 1-5) from 943 users on 1,682 movies, providing a rich source of user-movie interactions for analysis. The dataset is structured across three main files: u.data, which contains user-movie ratings with timestamps; u.item, which provides movie metadata including titles and release dates; and u.user, which contains demographic information about the users.

Our analysis revealed that the dataset is relatively clean, with no missing values in the core rating data. The ratings are distributed across a wide range of movies, with some movies receiving significantly more ratings than others, reflecting their popularity. The user base is diverse in terms of age, gender, and occupation, providing a representative sample for recommendation purposes. The temporal aspect of the ratings, captured through timestamps, allows for potential time-based analysis, though our current implementation focuses primarily on the rating values themselves. This comprehensive dataset enables us to build and evaluate different recommendation approaches effectively.

### **3. Methodology**

Our implementation follows a systematic approach to building recommendation systems using the MovieLens dataset. We begin with data preprocessing, which includes loading the dataset, handling missing values, and transforming the data into appropriate formats for analysis. A key transformation is the creation of a user-movie matrix, where rows represent users, columns represent movies, and cells contain the ratings given by users to movies. This matrix serves as the foundation for our collaborative filtering approaches.

For user-based collaborative filtering, we compute similarity between users using cosine similarity on their rating vectors. We then identify the most similar users to a target user and recommend movies that these similar users have rated highly but the target user hasn't yet watched. In item-based collaborative filtering, we transpose the user-movie matrix to compute similarity between movies based on their rating patterns across users. For a given movie, we can then identify the most similar movies to recommend.

The Pixie-inspired graph-based recommendation approach takes a different path. We construct a bipartite graph where users and movies are nodes, and edges represent ratings. We then implement a random walk algorithm that starts from either a user or a movie node and traverses the graph, counting the frequency of visits to movie nodes. Movies visited most frequently during these random walks are considered most relevant and are recommended to the user. This approach captures complex relationships in the data that might not be evident in matrix-based methods.

### **4. Results and Evaluation**

Our implementation of the three recommendation approaches yields interesting and diverse movie suggestions. The user-based collaborative filtering tends to recommend movies that align with the general taste profile of the user, based on the preferences of similar users. The item-based approach, meanwhile, focuses more on content similarity, recommending movies that share characteristics with those the user has already enjoyed.

The Pixie-inspired random walk approach offers perhaps the most diverse recommendations, as it can discover connections between movies that might not be immediately apparent through direct similarity measures. By traversing the user-movie graph, it can identify movies that are connected through complex chains of user preferences, potentially leading to more serendipitous discoveries.

While a formal evaluation of recommendation quality would require user feedback or hold-out testing, our implementation demonstrates the feasibility and potential of these approaches. Each method has its strengths: user-based filtering excels at capturing community preferences, item-based filtering is effective at finding similar content, and the graph-based approach can uncover hidden connections in the data. The choice between these methods would depend on specific application requirements, computational resources, and the desired balance between recommendation accuracy and diversity.

### **5. Conclusion and Future Work**

This project has demonstrated the implementation of three distinct approaches to movie recommendation using the MovieLens dataset. Each approach offers unique insights into user preferences and movie relationships, contributing to a comprehensive recommendation framework. The user-based and item-based collaborative filtering methods provide straightforward, interpretable recommendations based on similarity measures, while the Pixie-inspired random walk approach leverages graph structure to discover more complex relationships.

Future work could explore several avenues for enhancement. Incorporating additional features such as movie genres, user demographics, or temporal patterns could enrich the recommendation models. Hybrid approaches that combine multiple recommendation strategies might yield more robust results. Additionally, more sophisticated evaluation methods, such as A/B testing or offline metrics like precision and recall, could provide quantitative insights into recommendation quality.

The field of recommendation systems continues to evolve, with recent advances in deep learning and reinforcement learning offering new possibilities. Techniques such as neural collaborative filtering or sequence-aware recommendations could be explored to further improve recommendation quality. Ultimately, the goal remains to provide users with personalized, relevant, and diverse movie suggestions that enhance their entertainment experience.